# Stream processing modes
- Sagarmatha
- 2.0.0

## Context

   ![Pipeline](image/Spark_Streaming.png)

## Refrences
1. [Python API for Apache Spark](https://spark.apache.org/docs/latest/api/python/);

## Project creation and configuration
- `JAVA_HOME` must be set for PySpark to be imported correctly.
- `spark.sql.shuffle.partitions` sets the default number of partitions Spark creates when it redistributes data. For example during joins, grouping, aggregations, sorting or distinct etc.

In [ ]:
import os
import logging
import uuid
import shutil
import pyspark.sql.functions as F

from datetime import datetime, timezone
from time import time
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.streaming import StreamingQuery
from pathlib import Path
from delta import *

if "JAVA_HOME" not in os.environ:
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"

builder = (
    SparkSession
        .builder
        .master("local[*]")
        .appName("Sagarmatha")
        .config("spark.driver.memory", "2g")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

session = configure_spark_with_delta_pip(builder).getOrCreate()
session.conf.set("spark.sql.shuffle.partitions", "10")
session.conf.set("spark.sql.debug.maxToStringFields", "100")
session.conf.set("spark.sql.shuffle.partitions", "4")

DATABASE_NAME = "himalaya"
VOLUME_LOCATION = "spark-volume"
VOLUME_NAME = "tmp"
OPTIMISE_PERIOD = 5

## Datastore creation and configuration
With creation of database `himalaya` the datastore `spark-warehouse` will be automatically created too.

In [ ]:
session.sql(f"DROP DATABASE IF EXISTS {DATABASE_NAME} CASCADE")
session.sql(f"CREATE DATABASE IF NOT EXISTS {DATABASE_NAME}")
session.sql(f"USE DATABASE {DATABASE_NAME}")

Creating a volume for storing semistructured and unstructured data.

In [ ]:
shutil.rmtree(Path(VOLUME_LOCATION, VOLUME_NAME), ignore_errors=True)
Path(VOLUME_LOCATION, VOLUME_NAME).mkdir(parents=True, exist_ok=True)

## Checkpoints

**Checkpoints** are a fault-tolerance mechanism that periodically saves the state of a streaming to reliable, fault-tolerant storage. The mechanism allows to recover after a failure without losing data or recomputing everything from scratch.

![Spark Structured Streaming Checkpoint](image/Spark_Structured_Streaming_Checkpoint.png)

Function `get_random_checkpoint_location` creates if the not exists and returns random, unique checkpoint directory within the volume's location.

In [ ]:
def get_random_checkpoint_location(
    volume_location: str = VOLUME_LOCATION,
    volume_name: str = VOLUME_NAME,
    checkpoint_subdir: str = "",
) -> Path:
    """Create and return a new, unique checkpoint directory under the volume."""
    base = Path(volume_location, volume_name, "checkpoint")
    checkpoint_location = base / checkpoint_subdir / f"{datetime.now(timezone.utc):%Y%m%d_%H%M%S%f}_{uuid.uuid4().hex[:8]}"
    checkpoint_location.mkdir(parents=True, exist_ok=True)
    return checkpoint_location

## Analysis ⟶ Temporal aspects of data stream processing

Events in a streaming system do not flow instantaneously from their source to the processor. Network delays, buffering, and processing queues introduce gaps between when an event occurs and when it is actually handled. As a result, every event can be described by multiple timestamps:
- **Event time** represents the moment at which the event was originally created by the event source.
- **Ingestion time** represents the moment at which the event was received in the event stream within the event streaming/processing system.
- **Processing time** represents the moment at which the event processor processes the event. Potentially, this could be milliseconds, seconds, minutes, hours, months, etc. after the event time.

![Pipeline](image/Event_Time_Properties.svg)

Defining a **streaming data frame** with:
* default columns provided by the rate source: `timestamp, value`;
* derived columns: `event_time, ingestion_time, processing_time`.


In [ ]:
event_streaming_df = (
    session
        .readStream
        .format("rate")
        .option("rowsPerSecond", 5)
        .option("numPartitions", 1)
        .load()
        .withColumn("event_time", F.col("timestamp"))
        .withColumn("ingestion_time", F.expr("event_time + make_interval(0, 0, 0, 0, 0, 0, (rand()))"))
        .withColumn("processing_time", F.current_timestamp().cast("timestamp"))
)

event_streaming_df.printSchema()

Creating a **streaming query** with processing data every 1s:


In [ ]:
event_streaming_query = (
    event_streaming_df
        .writeStream
        .format("memory")
        .queryName("analysis")
        .trigger(processingTime="1 second")
        .option("checkpointLocation", get_random_checkpoint_location())
        .start()
)

In [ ]:
event_streaming_query.awaitTermination(20)
event_streaming_query.stop()

> Where does the streaming query write its data?

Analyse the order of events in the context of `event_time`, `ingestion_time`, and `processing_time`.
![Pipeline](image/Event_Time_Properties.png)

In [ ]:
session.sql("SELECT * FROM analysis").show(truncate=False)

> Why do the values in the `processing_time` column form groups?

### Processing 1 ⟶ Data stream MICRO-BATCH processing with persistent storage into DELTA TABLE.
Creating Delta Table `weather`.

In [ ]:
weather = (
    DeltaTable
        .createOrReplace(session)
        .tableName("weather")
        .addColumn("id", dataType="long", nullable=False)
        .addColumn("datetime", dataType="timestamp", nullable=False)
        .addColumn("air_temperature", dataType="real")
        .addColumn("humidity", dataType="integer")
        .addColumn("peak_id", dataType="integer")
        .partitionedBy("peak_id")
        .execute()
)

weather.detail().show(truncate=False, n=20)

Parameter `trigger` sets one of the following modes:
- `processingTime="n seconds"`
- `continuous="n seconds"`
- `availableNow=True`
- `once=True`


In [ ]:
def read_weather_measurements_stream(
    session: SparkSession,
    peak_id: int,
    rows_per_second: int = 10,
) -> DataFrame:
    """Return a streaming data frame object of synthetic weather measurements for the given peak."""
    return (
        session
            .readStream
            .format("rate")
            .option("rowsPerSecond", rows_per_second)
            .load()
            .withColumn("id", F.col("value"))
            .withColumn("datetime", F.col("timestamp"))
            .withColumn("air_temperature", F.round(10 - (F.rand() * 50), 2).cast("real"))
            .withColumn("humidity", (F.rand() * 100).cast("integer"))
            .withColumn("peak_id", F.lit(peak_id))
            .select("id", "datetime", "air_temperature", "humidity", "peak_id")
    )

def write_weather_measurements_stream(
    measurement_stream_df: DataFrame,
    random_checkpoint_location: Path,
) -> StreamingQuery:
    """Return a streaming query object representing an active job that writes measurements to the table."""
    return (
        measurement_stream_df
            .writeStream
            .format("delta")
            .option("checkpointLocation", random_checkpoint_location)
            .trigger(processingTime="5 seconds")
            .toTable("weather")
    )

def run_weather_measurements_stream(
    session: SparkSession,
    peak_id: int,
    period: int = 20,
) -> StreamingQuery:
    """Run a weather measurements stream for the given peak for PERIOD seconds, write to the table,
    then stop the query, remove its temporary checkpoint directory and return the terminated StreamingQuery."""
    random_checkpoint_location = get_random_checkpoint_location(checkpoint_subdir=str(peak_id))
    query = write_weather_measurements_stream(read_weather_measurements_stream(session, peak_id), random_checkpoint_location)

    try:
        terminated = query.awaitTermination(timeout=period)
        if not terminated:
            query.stop()
            query.awaitTermination(timeout=60)
    finally:
        if query.isActive:
            query.stop()
        if random_checkpoint_location.exists():
            shutil.rmtree(random_checkpoint_location)
            print("Checkpoints removed from location:", random_checkpoint_location)
        print("Streaming query with reading and writing ended. Status:", query.status)

    return query

Generowanie strumienia danych dla szczytu 1 oraz 2 przez okres 20 s.

In [ ]:
run_weather_measurements_stream(session, 1, 20)
run_weather_measurements_stream(session, 2, 20)

Wyświetlanie aktualnej liczby rekordów po rozpoczęciu generowania strumienia danych.

In [ ]:
session.sql("SELECT peak_id, COUNT(*) FROM weather GROUP BY peak_id ORDER BY 1 ASC").show(truncate=False);
session.sql("SELECT * FROM weather ORDER BY 2 DESC").show(truncate=False, n=100);

In [ ]:
weather.delete()

### Processing 2 ⟶ Data stream MICRO-BATCH UDA processing with persistent storage into DELTA TABLE

In [ ]:
logger = logging.getLogger("Batch streaming logger")
logger.setLevel(logging.INFO)

# Tworzenie funkcji wykonującej przetwarzanie (mikro)wsadowe (ang. (micro)batch processing).
def process_batch(
    batch_output: DataFrame,
    batch_id: int,
) -> None:
    """
    Parametery:
         batch_output: ramka danych
         batch_id: numer operacji przetwarzania (mikro)wsadowego
    """
    logger.info(f"Przetwarzanie (mikro)wsadowe nr {batch_id}.")
    print("Przetwarzanie (mikro)wsadowe nr %s." % batch_id)

    weather = DeltaTable.forName(session, "weather")

    weather \
        .alias("target") \
        .merge(
            batch_output.alias("source"),
            "target.datetime = source.datetime" or "target.peak_id = source.peak_id"
        ) \
        .whenNotMatchedInsertAll() \
        .execute()

def write_weather_measurements_stream_batch(
    measurement_stream_df: DataFrame,
    random_checkpoint_location: Path,
) -> StreamingQuery:
    """Return a streaming query object representing an active job that performs micro-batch function which writes measurements to the table."""
    return (
        measurement_stream_df
            .writeStream
            .format("delta")
            .option("checkpointLocation", random_checkpoint_location)
            .trigger(processingTime="5 seconds")
            .foreachBatch(process_batch)
            .start()
    )

def run_weather_measurements_stream_batch(
    session: SparkSession,
    peak_id: int,
    period: int = 20,
) -> StreamingQuery:
    """Run a weather measurements stream for the given peak for PERIOD seconds, write to the table,
    then stop the query, remove its temporary checkpoint directory and return the terminated StreamingQuery."""
    random_checkpoint_location = get_random_checkpoint_location(checkpoint_subdir=str(peak_id))
    query = write_weather_measurements_stream_batch(read_weather_measurements_stream(session, peak_id), random_checkpoint_location)

    try:
        terminated = query.awaitTermination(timeout=period)
        if not terminated:
            query.stop()
            query.awaitTermination(timeout=60)
    finally:
        if query.isActive:
            query.stop()
        if random_checkpoint_location.exists():
            shutil.rmtree(random_checkpoint_location)
            print("Checkpoints removed from location:", random_checkpoint_location)
        print("Streaming query with reading and writing ended. Status:", query.status)

    return query

Generowanie strumienia danych dla szczytu 1 oraz 2 przez okres 30 s.

In [ ]:
run_weather_measurements_stream_batch(session, 1, 30)
run_weather_measurements_stream_batch(session, 2, 30)

Wyświetlanie aktualnej liczby rekordów po rozpoczęciu generowania strumienia danych.

In [ ]:
session.sql("SELECT peak_id, COUNT(*) FROM weather GROUP BY peak_id ORDER BY 1 ASC").show(truncate=False);
session.sql("SELECT * FROM weather ORDER BY 2 DESC").show(truncate=False, n=100);

### Processing 3 ⟶ Data stream MICRO-BATCH UDA processing with OPTIMIZATION and persistent storage into DELTA TABLE

In [ ]:
logger = logging.getLogger("Optimise logger")
logger.setLevel(logging.INFO)

# Tworzenie funkcji optymalizującej tabelę Delta Lake poprzez porządkowanie danych ZORDER.
def optimise_table(
    table_name: str,
    column_name: str,
) -> None:
    """
    Parametery:
         table_name: optymalizowana tabela
         column_name: kolumna / lista kolumn wg których zostanie wyokonana operacja ZORDER BY
    """
    start = time()
    print("Optymalizowanie tabeli %s." % table_name)
    logger.info(f"Optymalizowanie tabeli {table_name}.")
    query = f"OPTIMIZE {table_name} ZORDER BY ({column_name})"
    session.sql(query)
    end = time()
    print("Tabela zoptymalizowana/uporządkowana w czasie %s s." % (end - start))
    logger.info(f"Tabela zoptymalizowana/uporządkowana w czasie {end - start} s.")

In [ ]:
logger = logging.getLogger("Batch streaming logger")
logger.setLevel(logging.INFO)

# Tworzenie funkcji wykonującej przetwarzanie (mikro)wsadowe (ang. batch processing) z optymalizacją
def process_batch(
    batch_output: DataFrame,
    batch_id: int,
) -> None:
    """
    Parametery:
         batch_output: ramka danych
         batch_id: numer operacji przetwarzania (mikro)wsadowego
    """
    logger.info(f"Przetwarzanie (mikro)wsadowe nr {batch_id}.")
    print("Przetwarzanie (mikro)wsadowe nr %s." % batch_id, flush=True)

    weather = DeltaTable.forName(session, "weather")

    weather \
        .alias("target") \
        .merge(
            batch_output.alias("source"),
            "target.datetime = source.datetime" or "target.peak_id = source.peak_id"
        ) \
        .whenNotMatchedInsertAll() \
        .execute()

    if batch_id % OPTIMISE_PERIOD == 0:
        optimise_table("weather", "humidity")

Generowanie strumienia danych pogodowych z przetwarzaniem (mikro)wsadowym oraz optymalizacją dla szczytu 3 i 4 przez okres 50 s.

In [ ]:
run_weather_measurements_stream_batch(session, 3, 50)
run_weather_measurements_stream_batch(session, 4, 50)

Wyświetlanie aktualnej liczby rekordów po rozpoczęciu generowania strumienia danych.

In [ ]:
session.sql("SELECT peak_id, COUNT(*) FROM weather GROUP BY peak_id ORDER BY 1 ASC").show(truncate=False);
session.sql("SELECT * FROM weather ORDER BY 2 DESC").show(truncate=False, n=100);

### Processing 4 ⟶ Data stream CONTINOUS processing

In [ ]:
query = (
    session.readStream
        .format("rate")
        .option("rowsPerSecond", 10)
        .option("numPartitions", 2)
        .load()
        .selectExpr("value", "value * 2 AS doubled")
        .where("value % 4 = 0")
        .writeStream
        .format("console")
        .trigger(continuous="1 second")
        .start()
)

query.awaitTermination(20)
query.stop()

### Processing 5 ⟶ Data stream REAL TIME processing

In [ ]:
# NOT RUNNABLE - KAFKA NEEDED
query = (
    session
        .readStream
        .format("rate")
        .option("rowsPerSecond", 10)
        .option("numPartitions", 2)
        .load()
        .selectExpr("value", "value * 2 AS doubled")
        .where("value % 2 = 0")
        .writeStream
        .format("console")
        .outputMode("update")
        .trigger(realTime="5 second")
        .start()
)
query.awaitTermination(20)
query.stop()

## Additions
1. Viewing list of databases;
2. Creation of the delta table;
3. Viewing version of the table in different moments of existence with usage of **time travel** mechanism.

In [ ]:
session.catalog.listDatabases()
weather.write.format("delta").saveAsTable("weather")
session.sql("DESCRIBE HISTORY weather").show()

In [ ]:
# Zdefiniowanie lokalizacji tabeli w systemie plików HDFS via nazwa tabeli zarejestrowanej w katalogu (HIVE METASTORE, IN-MEMORY, UNITY CATALOG).
weather_location = session.sql("DESCRIBE DETAIL weather").select("location").first()["location"]

# Wyświetlenie lokalizacji.
display(weather_location)
session.sql("DESCRIBE DETAIL weather").show(truncate=False)

In [ ]:
print(session.conf.get("spark.sql.catalogImplementation"))
session.sql("SHOW CATALOGS").show()
session.sql("SHOW NAMESPACES").show()
session.sql("SELECT current_catalog(), current_database()").show()

In [ ]:
'''
def run_stream(
    query,
    period=20,
) -> None:
    """Process data of a streaming query."""
    if query is None:
        return
    if not query.isActive:
        return
    try:
        terminated = query.awaitTermination(timeout=period)
        if not terminated:
            query.stop()
            query.awaitTermination(timeout=60)
    finally:
        if query.isActive:
            query.stop()
        print("Streaming query with reading and writing ended. Status:", query.status)

run_stream(event_streaming_query, period=20)
'''